In [ ]:
#Cargar librerias
import pandas as pd
import numpy as np

#Cargar los datos
df=pd.read_csv("Datos/Transformados/clustering_final.csv")

In [ ]:
#OBJETIVO 1: Simulación de amortizaciones bajo sistemas FRANCÉS y ALEMÁN
#Cálculo de intereses totales, cuota máxima y saldo pendiente a los 12 meses

#Columnas del DataFrame original que se utilizarán para los cálculos
cols = ["ID", "Monto_Inicial", "Duracion", "Ratio_Interes"]

#Conversión del ratio de interés de porcentaje a decimal para cálculos financieros
df["interes_anual"] = df["Ratio_Interes"] / 100


def amortizacion_frances(capital, interes_anual, meses):
    """
    Calcula la tabla de amortización completa para el sistema francés (cuota constante)
    """
    #Cálculo de la tasa de interés mensual
    i = interes_anual / 12
    #Fórmula de la cuota constante del sistema francés
    cuota = capital * (i * (1 + i)**meses) / ((1 + i)**meses - 1)

    saldo = capital
    tabla = []

    #Generación mes a mes de la tabla de amortización
    for mes in range(1, meses + 1):
        interes = saldo * i
        amort = cuota - interes
        saldo -= amort

        tabla.append([mes, cuota, interes, amort, max(saldo, 0)])

    return pd.DataFrame(tabla, columns=["mes","cuota","interes","amort","saldo"])


def amortizacion_aleman(capital, interes_anual, meses):
    """
    Calcula la tabla de amortización completa para el sistema alemán (amortización constante)
    """
    #Cálculo de la tasa de interés mensual
    i = interes_anual / 12
    #Amortización constante de capital en cada período
    amort_const = capital / meses

    saldo = capital
    tabla = []

    #Generación mes a mes de la tabla de amortización
    for mes in range(1, meses + 1):
        interes = saldo * i
        cuota = amort_const + interes
        saldo -= amort_const

        tabla.append([mes, cuota, interes, amort_const, max(saldo, 0)])

    return pd.DataFrame(tabla, columns=["mes","cuota","interes","amort","saldo"])


#DataFrame para almacenar los resultados comparativos de cada préstamo
resultados = []

#Iteración sobre cada fila del DataFrame original para calcular métricas
for _, row in df.iterrows():
    capital = row["Monto_Inicial"]
    meses = int(row["Duracion"])
    interes = row["interes_anual"]

    #Cálculo de tablas para ambos sistemas
    f = amortizacion_frances(capital, interes, meses)
    a = amortizacion_aleman(capital, interes, meses)

    #Almacenamiento de métricas relevantes de cada sistema
    resultados.append({
        "intereses_frances": f["interes"].sum(),
        "intereses_aleman": a["interes"].sum(),
        "cuota_max_frances": f["cuota"].max(),
        "cuota_max_aleman": a["cuota"].max(),
        "saldo_12_frances": f.loc[f["mes"]==12,"saldo"].values[0] if meses>=12 else np.nan,
        "saldo_12_aleman": a.loc[a["mes"]==12,"saldo"].values[0] if meses>=12 else np.nan
    })

#Conversión de resultados a DataFrame para análisis
res = pd.DataFrame(resultados)

#Visualización de resultados comparativos medios
print("Intereses medios:")
print("Frances:", round(res["intereses_frances"].mean(),2))
print("Aleman :", round(res["intereses_aleman"].mean(),2))

print("\nCuota maxima media:")
print("Frances:", round(res["cuota_max_frances"].mean(),2))
print("Aleman :", round(res["cuota_max_aleman"].mean(),2))

print("\nSaldo pendiente medio tras 12 meses:")
print("Frances:", round(res["saldo_12_frances"].mean(),2))
print("Aleman :", round(res["saldo_12_aleman"].mean(),2))

Intereses medios:
Frances: 4567.95
Aleman : 4160.57

Cuota maxima media:
Frances: 884.33
Aleman : 983.04

Saldo pendiente medio tras 12 meses:
Frances: 11542.25
Aleman : 10853.24


In [ ]:
#OBJETIVO 2: Valor presente de las cuotas pendientes a mitad del préstamo
#Sistema seleccionado: ALEMÁN

def valor_presente_mitad_aleman(capital, interes_anual, meses):
    """
    Calcula el valor presente en la mitad del préstamo (mes = duracion // 2)
    de todas las cuotas pendientes desde ese momento hasta el final.
    Sistema de amortización: ALEMÁN
    """
    i = interes_anual / 12
    n = int(meses)
    amort_const = capital / n
    
    mitad = n // 2  #mes de la mitad (división entera)
    
    #Generar todas las cuotas del préstamo (sistema alemán)
    cuotas = []
    saldo = capital
    for t in range(1, n + 1):
        interes = saldo * i
        cuota = amort_const + interes
        cuotas.append(cuota)
        saldo -= amort_const
    
    #Valor presente en el momento 'mitad' de las cuotas desde mitad+1 hasta n
    vp = 0
    for k in range(mitad, n):  # k es índice 0-based en la lista 'cuotas'
        vp += cuotas[k] / ((1 + i) ** (k - mitad + 1))
    
    return vp


#Aplicar la función a cada préstamo del DataFrame
df["VP_mitad_aleman"] = df.apply(
    lambda row: valor_presente_mitad_aleman(
        row["Monto_Inicial"],
        row["interes_anual"],
        row["Duracion"]
    ),
    axis=1
)

#Resultados
print("\n" + "="*60)
print("OBJETIVO 2: VALOR PRESENTE A MITAD DEL PRÉSTAMO (SISTEMA ALEMÁN)")
print("="*60)
print(df[["ID", "Monto_Inicial", "Duracion", "VP_mitad_aleman"]].head(10))
print("\nEstadísticos descriptivos:")
print(df["VP_mitad_aleman"].describe().round(2))


OBJETIVO 2: VALOR PRESENTE A MITAD DEL PRÉSTAMO (SISTEMA ALEMÁN)
       ID  Monto_Inicial  Duracion  VP_mitad_aleman
0  S97R7X           5000        48           2500.0
1  RLGTBY          37278        12          18639.0
2  SKE2P9          44532        60          22266.0
3  E2FB1D          23752        24          11876.0
4  TKSCGH          28440        12          14220.0
5  8CI0EZ          34451        48          17225.5
6  NBL3DD          21299        12          10649.5
7  ABRPBC           5000        36           2500.0
8  JU5UY4           7310        60           3655.0
9  6789JG          49090        36          24545.0

Estadísticos descriptivos:
count    50844.00
mean      9978.07
std       6834.84
min       2500.00
25%       4917.00
50%       8356.25
75%      13105.00
max      40000.00
Name: VP_mitad_aleman, dtype: float64


In [ ]:
# Comparación con saldo nominal a mitad del préstamo (opcional)
def saldo_nominal_mitad_aleman(capital, interes_anual, meses):
    """Calcula el saldo pendiente nominal en la mitad del préstamo (alemán)"""
    n = int(meses)
    mitad = n // 2
    return capital - (capital / n) * mitad

df["saldo_nominal_mitad"] = df.apply(
    lambda row: saldo_nominal_mitad_aleman(
        row["Monto_Inicial"],
        row["interes_anual"],
        row["Duracion"]
    ),
    axis=1
)

print("\nComparación saldo nominal vs valor presente (media):")
print(f"Saldo nominal a mitad: {df['saldo_nominal_mitad'].mean():.2f} €")
print(f"Valor presente a mitad: {df['VP_mitad_aleman'].mean():.2f} €")
print(f"Diferencia (ahorro por descuento): {df['saldo_nominal_mitad'].mean() - df['VP_mitad_aleman'].mean():.2f} €")


Comparación saldo nominal vs valor presente (media):
Saldo nominal a mitad: 9978.07 €
Valor presente a mitad: 9978.07 €
Diferencia (ahorro por descuento): 0.00 €
